In [92]:
import os
import re
import pdfplumber
import pandas as pd
import numpy as np

try:
    from tqdm.notebook import tqdm
except:
    from tqdm import tqdm

pdf_folder = "../pdfs"
output_file_tabla2 = "tabla_2.csv"

pdf_files = sorted([f for f in os.listdir(pdf_folder) if f.lower().endswith(".pdf")])
pdf_files

['09_abril_2026.pdf',
 '10_abril_2026.pdf',
 '12_diciembre_2025.pdf',
 '13_abril_2026.pdf',
 '13_febrero_2026.pdf',
 '14_abril_2026.pdf',
 '15_abril_2026.pdf',
 '15_diciembre_2025.pdf',
 '16_abril_2026.pdf',
 '16_diciembre_2025.pdf',
 '16_enero_2026.pdf',
 '16_setiembre_2025.pdf',
 '17_abril_2026.pdf',
 '17_diciembre_2025.pdf',
 '17_febrero_2026.pdf',
 '17_noviembre_2025.pdf',
 '17_setiembre_2025.pdf',
 '18_diciembre_2025.pdf',
 '18_febrero_2026.pdf',
 '18_marzo_2026.pdf',
 '18_noviembre_2025.pdf',
 '18_setiembre_2025.pdf',
 '19_diciembre_2025.pdf',
 '19_enero_2026.pdf',
 '19_febrero_2026.pdf',
 '19_marzo_2026.pdf',
 '19_noviembre_2025.pdf',
 '19_setiembre_2025.pdf',
 '20_abril_2026.pdf',
 '20_enero_2026.pdf',
 '20_febrero_2026.pdf',
 '20_marzo_2026.pdf',
 '20_noviembre_2025.pdf',
 '20_octubre_2025.pdf',
 '21_abril_2026.pdf',
 '21_enero_2026.pdf',
 '21_noviembre_2025.pdf',
 '21_octubre_2025.pdf',
 '22_abril_2026.pdf',
 '22_diciembre_2025.pdf',
 '22_enero_2026.pdf',
 '22_octubre_2025.pd

In [93]:
MESES = {
    "ene": 1, "feb": 2, "mar": 3, "abr": 4, "may": 5, "jun": 6,
    "jul": 7, "ago": 8, "sep": 9, "oct": 10, "nov": 11, "dic": 12
}

COLUMNAS_TABLA2 = [
    "fecha", "anio", "mes_txt", "mes_num", "dia",
    "lima", "sierra", "costa_norte", "selva", "costa_sur", "total", "pdf_file"
]

PATRON_FECHA_REPORTE = re.compile(r"Fecha\s+(\d{1,2})/(\d{1,2})/(\d{4})", re.IGNORECASE)

PATRON_FILA = re.compile(
    r"^\s*(ene|feb|mar|abr|may|jun|jul|ago|sep|oct|nov|dic)\s+(\d{1,2})\s+(.+)$",
    re.IGNORECASE
)

def normalizar_texto(texto):
    if not texto:
        return ""
    texto = texto.replace("\xa0", " ")
    lineas = texto.split("\n")
    lineas = [re.sub(r"\s+", " ", x).strip() for x in lineas if x.strip()]
    return "\n".join(lineas)

def obtener_fecha_reporte(texto):
    m = PATRON_FECHA_REPORTE.search(texto)
    if m:
        return tuple(map(int, m.groups()))
    return None, None, None

def aislar_tabla2(texto):
    texto = normalizar_texto(texto)

    patron = re.compile(
        r"OFERTA\s+DEL\s+HUEVO\s+DE\s+PRIMERA\s+SEGUN\s+MACROREGION\s*\(toneladas\)"
        r".*?"
        r"Fecha\s+LIMA\s+Sierra\s+Costa\s+Norte\s+Selva\s+Costa\s+Sur\s+TOTAL"
        r"(.*?)"
        r"Fuente:\s+Empresas\s+productoras\s+de\s+huevo\.?",
        re.DOTALL
    )

    m = patron.search(texto)
    if m:
        return normalizar_texto(m.group(1))

    return ""

def convertir_numero(x):
    x = str(x).strip()
    if x in ["", "-", "#N/A", "#n/a"]:
        return np.nan
    return float(x.replace(",", ""))

def anio_de_fila(mes_fila, mes_reporte, anio_reporte):
    if mes_reporte == 1 and mes_fila == 12:
        return anio_reporte - 1
    return anio_reporte

def parsear_tabla2(texto_tabla2, texto_pagina, pdf_file):
    _, mes_rep, anio_rep = obtener_fecha_reporte(texto_pagina)

    filas = []

    if not texto_tabla2 or anio_rep is None:
        return pd.DataFrame(columns=COLUMNAS_TABLA2)

    for linea in texto_tabla2.split("\n"):
        linea = re.sub(r"\s+", " ", linea.strip())

        if linea.lower().startswith("total"):
            continue

        m = PATRON_FILA.match(linea)
        if not m:
            continue

        mes_txt, dia_txt, resto = m.groups()
        mes_txt = mes_txt.lower()
        dia = int(dia_txt)
        mes_num = MESES.get(mes_txt)

        valores = resto.split()

        if len(valores) < 6:
            continue

        valores = valores[:6]

        try:
            lima, sierra, costa_norte, selva, costa_sur, total = [convertir_numero(v) for v in valores]
        except:
            continue

        anio_fila = anio_de_fila(mes_num, mes_rep, anio_rep)

        fecha = pd.to_datetime(
            {"year": [anio_fila], "month": [mes_num], "day": [dia]},
            errors="coerce"
        )[0]

        if pd.isna(fecha):
            continue

        filas.append({
            "fecha": fecha,
            "anio": anio_fila,
            "mes_txt": mes_txt,
            "mes_num": mes_num,
            "dia": dia,
            "lima": lima,
            "sierra": sierra,
            "costa_norte": costa_norte,
            "selva": selva,
            "costa_sur": costa_sur,
            "total": total,
            "pdf_file": pdf_file
        })

    df = pd.DataFrame(filas, columns=COLUMNAS_TABLA2)

    if df.empty:
        return pd.DataFrame(columns=COLUMNAS_TABLA2)

    df = df.drop_duplicates()

    if df["fecha"].duplicated().any():
        cols_num = ["lima", "sierra", "costa_norte", "selva", "costa_sur", "total"]
        df["n_no_nulos"] = df[cols_num].notna().sum(axis=1)
        df = (
            df.sort_values(["fecha", "n_no_nulos"], ascending=[True, False])
              .drop_duplicates(subset=["fecha"], keep="first")
              .drop(columns="n_no_nulos")
        )

    return df.sort_values("fecha").reset_index(drop=True)

In [94]:
if os.path.exists(output_file_tabla2):
    df_hist_tabla2 = pd.read_csv(output_file_tabla2, parse_dates=["fecha"])

    # limpiar histórico: eliminar filas sin pdf_file
    df_hist_tabla2 = df_hist_tabla2[df_hist_tabla2["pdf_file"].notna()].copy()
    df_hist_tabla2["pdf_file"] = df_hist_tabla2["pdf_file"].astype(str).str.strip()

    # guardar histórico limpio
    df_hist_tabla2.to_csv(output_file_tabla2, index=False)

    pdfs_procesados = set(df_hist_tabla2["pdf_file"].unique())
else:
    df_hist_tabla2 = pd.DataFrame(columns=COLUMNAS_TABLA2)
    pdfs_procesados = set()

pdf_files_nuevos = [f for f in pdf_files if f not in pdfs_procesados]

print("PDFs encontrados:", len(pdf_files))
print("PDFs ya procesados:", len(pdfs_procesados))
print("PDFs nuevos:", len(pdf_files_nuevos))

pdf_files_nuevos[:10]

PDFs encontrados: 84
PDFs ya procesados: 84
PDFs nuevos: 0


[]

In [95]:
registros_texto = []
errores_extraccion = []

for file_name in tqdm(pdf_files_nuevos, desc="Extrayendo texto de PDFs", unit="pdf"):
    pdf_path = os.path.join(pdf_folder, file_name)

    try:
        with pdfplumber.open(pdf_path) as pdf:
            texto_pagina = normalizar_texto(pdf.pages[0].extract_text() or "")

        registros_texto.append({
            "pdf_file": file_name,
            "texto_pagina": texto_pagina
        })

    except Exception as e:
        errores_extraccion.append({
            "pdf_file": file_name,
            "error": str(e)
        })

df_textos_tabla2 = pd.DataFrame(registros_texto)
df_errores_extraccion_tabla2 = pd.DataFrame(errores_extraccion)

display(df_textos_tabla2.head())
display(df_errores_extraccion_tabla2)

Extrayendo texto de PDFs: 0pdf [00:00, ?pdf/s]

""


""


In [96]:
lista_dfs = []
errores_parseo = []

for _, row in df_textos_tabla2.iterrows():
    file_name = row["pdf_file"]
    texto_pagina = row["texto_pagina"]

    try:
        texto_tabla2 = aislar_tabla2(texto_pagina)
        df_tmp = parsear_tabla2(texto_tabla2, texto_pagina, file_name)
        lista_dfs.append(df_tmp)

    except Exception as e:
        errores_parseo.append({
            "pdf_file": file_name,
            "error": str(e)
        })

df_nuevos_tabla2 = (
    pd.concat(lista_dfs, ignore_index=True)
    if lista_dfs else pd.DataFrame(columns=COLUMNAS_TABLA2)
)

df_errores_parseo_tabla2 = pd.DataFrame(errores_parseo)

df_total_tabla2 = pd.concat([df_hist_tabla2, df_nuevos_tabla2], ignore_index=True)

df_total_tabla2 = df_total_tabla2.drop_duplicates()

if not df_total_tabla2.empty:
    df_total_tabla2 = (
        df_total_tabla2
        .drop_duplicates(subset=["fecha", "pdf_file"], keep="first")
        .sort_values(["fecha", "pdf_file"])
        .reset_index(drop=True)
    )

df_total_tabla2.to_csv(output_file_tabla2, index=False)

print("Guardado en:", output_file_tabla2)
print("Filas nuevas:", len(df_nuevos_tabla2))
print("Filas totales:", len(df_total_tabla2))

display(df_nuevos_tabla2.head())
display(df_errores_parseo_tabla2)

Guardado en: tabla_2.csv
Filas nuevas: 0
Filas totales: 1147


C:\Users\JHOSSEP\AppData\Local\Temp\ipykernel_17952\3163383886.py:26: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_total_tabla2 = pd.concat([df_hist_tabla2, df_nuevos_tabla2], ignore_index=True)


,fecha,anio,mes_txt,mes_num,dia,lima,sierra,costa_norte,selva,costa_sur,total,pdf_file


""


In [97]:
cols_region = ["lima", "sierra", "costa_norte", "selva", "costa_sur"]

if df_total_tabla2.empty:
    print("No se encontraron datos para tabla 2.")
else:
    duplicados_fecha = df_total_tabla2[
        df_total_tabla2.duplicated(subset=["fecha"], keep=False)
    ].sort_values("fecha")

    df_check = df_total_tabla2.copy()
    df_check["suma_regiones"] = df_check[cols_region].fillna(0).sum(axis=1)
    df_check["diff_total"] = (df_check["total"] - df_check["suma_regiones"]).abs()

    inconsistentes = df_check[df_check["diff_total"] > 1]

    display(duplicados_fecha)
    display(inconsistentes[[
        "fecha", "lima", "sierra", "costa_norte", "selva", "costa_sur",
        "total", "suma_regiones", "diff_total", "pdf_file"
    ]])

,fecha,anio,mes_txt,mes_num,dia,lima,sierra,costa_norte,selva,costa_sur,total,pdf_file
1,2025-09-04,2025,sep,9,4,156664.0,53435.0,0.0,3040.0,15899.0,229038.0,16_setiembre_2025.pdf
2,2025-09-04,2025,sep,9,4,156664.0,53435.0,0.0,3040.0,15899.0,229038.0,17_setiembre_2025.pdf
3,2025-09-05,2025,sep,9,5,103191.0,62557.0,0.0,9598.0,9944.0,185290.0,16_setiembre_2025.pdf
4,2025-09-05,2025,sep,9,5,103191.0,62557.0,0.0,9598.0,9944.0,185290.0,17_setiembre_2025.pdf
5,2025-09-05,2025,sep,9,5,103191.0,62557.0,0.0,9598.0,9944.0,185290.0,18_setiembre_2025.pdf
...,...,...,...,...,...,...,...,...,...,...,...,...
1141,2026-04-21,2026,abr,4,21,136093.0,121187.0,6215.0,11891.0,0.0,275386.0,22_abril_2026.pdf
1143,2026-04-21,2026,abr,4,21,136093.0,121187.0,6215.0,11891.0,0.0,275386.0,26_abril_2026.pdf
1145,2026-04-22,2026,abr,4,22,75818.0,32195.0,0.0,0.0,0.0,108013.0,23_abril_2026.pdf
1144,2026-04-22,2026,abr,4,22,NaN,NaN,NaN,NaN,NaN,NaN,22_abril_2026.pdf


,fecha,lima,sierra,costa_norte,selva,costa_sur,total,suma_regiones,diff_total,pdf_file
734,2026-02-05,90796.0,72881.0,9238.0,3955.0,0.0,176872.0,176870.0,2.0,13_febrero_2026.pdf
735,2026-02-05,90796.0,72881.0,9238.0,3955.0,0.0,176872.0,176870.0,2.0,17_febrero_2026.pdf
736,2026-02-05,90796.0,72881.0,9238.0,3955.0,0.0,176872.0,176870.0,2.0,18_febrero_2026.pdf


# Revisamo la tabla

In [98]:
df_total_tabla2

,fecha,anio,mes_txt,mes_num,dia,lima,sierra,costa_norte,selva,costa_sur,total,pdf_file
0,2025-09-03,2025,sep,9,3,286212.0,115589.0,0.0,10368.0,6149.0,418318.0,16_setiembre_2025.pdf
1,2025-09-04,2025,sep,9,4,156664.0,53435.0,0.0,3040.0,15899.0,229038.0,16_setiembre_2025.pdf
2,2025-09-04,2025,sep,9,4,156664.0,53435.0,0.0,3040.0,15899.0,229038.0,17_setiembre_2025.pdf
3,2025-09-05,2025,sep,9,5,103191.0,62557.0,0.0,9598.0,9944.0,185290.0,16_setiembre_2025.pdf
4,2025-09-05,2025,sep,9,5,103191.0,62557.0,0.0,9598.0,9944.0,185290.0,17_setiembre_2025.pdf
...,...,...,...,...,...,...,...,...,...,...,...,...
1142,2026-04-21,2026,abr,4,21,136093.0,121187.0,6215.0,11891.0,0.0,275386.0,23_abril_2026.pdf
1143,2026-04-21,2026,abr,4,21,136093.0,121187.0,6215.0,11891.0,0.0,275386.0,26_abril_2026.pdf
1144,2026-04-22,2026,abr,4,22,NaN,NaN,NaN,NaN,NaN,NaN,22_abril_2026.pdf
1145,2026-04-22,2026,abr,4,22,75818.0,32195.0,0.0,0.0,0.0,108013.0,23_abril_2026.pdf


In [99]:
df_total = df_total_tabla2.copy()

## A veces datos de el pdf se corrigen en días posteriores, entonces para la misma fecha tomamos el último pdf

In [100]:
# CREAMOS FECHA pdf
import pandas as pd

# Diccionario de meses
meses_map = {
    'enero':1, 'febrero':2, 'marzo':3, 'abril':4,
    'mayo':5, 'junio':6, 'julio':7, 'agosto':8,
    'setiembre':9, 'septiembre':9, 'octubre':10,
    'noviembre':11, 'diciembre':12
}

# Extraer partes del nombre del PDF
df_total[['dia_pdf', 'mes_pdf_txt', 'anio_pdf']] = df_total['pdf_file'] \
    .str.replace('.pdf', '', regex=False) \
    .str.split('_', expand=True)

# Convertir mes a número
df_total['mes_pdf'] = df_total['mes_pdf_txt'].map(meses_map)

# Crear fecha_pdf
df_total['fecha_pdf'] = pd.to_datetime(
    dict(year=df_total['anio_pdf'].astype(int),
         month=df_total['mes_pdf'],
         day=df_total['dia_pdf'].astype(int))
)

Nos quedamos con el dato más reciente

In [101]:
# p sea la fecha las ponemos en orden,  y leugo ordenamos por fecha pdf, la más reciente (primera o última ) es la que queda
df_total= df_total.sort_values(['fecha', 'fecha_pdf']) # ordena en ascendente por defecto (la íultima es la mas reciente)

df_final = df_total.drop_duplicates(subset=['fecha'], keep='last')

In [102]:
df_final

,fecha,anio,mes_txt,mes_num,dia,lima,sierra,costa_norte,selva,costa_sur,total,pdf_file,dia_pdf,mes_pdf_txt,anio_pdf,mes_pdf,fecha_pdf
0,2025-09-03,2025,sep,9,3,286212.0,115589.0,0.0,10368.0,6149.0,418318.0,16_setiembre_2025.pdf,16,setiembre,2025,9,2025-09-16
2,2025-09-04,2025,sep,9,4,156664.0,53435.0,0.0,3040.0,15899.0,229038.0,17_setiembre_2025.pdf,17,setiembre,2025,9,2025-09-17
5,2025-09-05,2025,sep,9,5,103191.0,62557.0,0.0,9598.0,9944.0,185290.0,18_setiembre_2025.pdf,18,setiembre,2025,9,2025-09-18
9,2025-09-06,2025,sep,9,6,97955.0,72622.0,0.0,6363.0,4969.0,181909.0,19_setiembre_2025.pdf,19,setiembre,2025,9,2025-09-19
13,2025-09-07,2025,sep,9,7,24295.0,0.0,0.0,0.0,0.0,24295.0,19_setiembre_2025.pdf,19,setiembre,2025,9,2025-09-19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1131,2026-04-18,2026,abr,4,18,100437.0,65127.0,20421.0,6263.0,0.0,192249.0,26_abril_2026.pdf,26,abril,2026,4,2026-04-26
1136,2026-04-19,2026,abr,4,19,28155.0,0.0,0.0,0.0,0.0,28155.0,26_abril_2026.pdf,26,abril,2026,4,2026-04-26
1140,2026-04-20,2026,abr,4,20,149288.0,127808.0,40762.0,6150.0,0.0,324008.0,26_abril_2026.pdf,26,abril,2026,4,2026-04-26
1143,2026-04-21,2026,abr,4,21,136093.0,121187.0,6215.0,11891.0,0.0,275386.0,26_abril_2026.pdf,26,abril,2026,4,2026-04-26


In [103]:
# pdf_file no nos sirve lo borramos ( y hace que las flas sean diferentes solo por esa columna)
df_total_tabla2 = df_total_tabla2.drop(columns=["pdf_file"])

In [104]:
df_total_tabla2.duplicated().sum()

854

In [105]:
# eliminamos duplicados
df_total_tabla2 = df_total_tabla2.drop_duplicates()

In [106]:
df_total_tabla2

,fecha,anio,mes_txt,mes_num,dia,lima,sierra,costa_norte,selva,costa_sur,total
0,2025-09-03,2025,sep,9,3,286212.0,115589.0,0.0,10368.0,6149.0,418318.0
1,2025-09-04,2025,sep,9,4,156664.0,53435.0,0.0,3040.0,15899.0,229038.0
3,2025-09-05,2025,sep,9,5,103191.0,62557.0,0.0,9598.0,9944.0,185290.0
6,2025-09-06,2025,sep,9,6,97955.0,72622.0,0.0,6363.0,4969.0,181909.0
10,2025-09-07,2025,sep,9,7,24295.0,0.0,0.0,0.0,0.0,24295.0
...,...,...,...,...,...,...,...,...,...,...,...
1132,2026-04-19,2026,abr,4,19,28155.0,0.0,0.0,0.0,0.0,28155.0
1137,2026-04-20,2026,abr,4,20,149288.0,127808.0,40762.0,6150.0,0.0,324008.0
1141,2026-04-21,2026,abr,4,21,136093.0,121187.0,6215.0,11891.0,0.0,275386.0
1144,2026-04-22,2026,abr,4,22,NaN,NaN,NaN,NaN,NaN,NaN


Duplicados por fecha


In [107]:
# ahora revisamos duplucados por fecha
revisar = df_total_tabla2[df_total_tabla2.duplicated(subset=["fecha"], keep=False)].sort_values("fecha")
revisar.head(30)

,fecha,anio,mes_txt,mes_num,dia,lima,sierra,costa_norte,selva,costa_sur,total
64,2025-09-16,2025,sep,9,16,NaN,NaN,NaN,NaN,NaN,NaN
65,2025-09-16,2025,sep,9,16,143524.0,132560.0,20358.0,1091.0,0.0,297533.0
73,2025-09-17,2025,sep,9,17,NaN,NaN,NaN,NaN,NaN,NaN
74,2025-09-17,2025,sep,9,17,132460.0,89203.0,10312.0,4144.0,0.0,236119.0
82,2025-09-18,2025,sep,9,18,NaN,NaN,NaN,NaN,NaN,NaN
83,2025-09-18,2025,sep,9,18,109177.0,52134.0,23605.0,13459.0,0.0,198374.0
84,2025-09-18,2025,sep,9,18,109177.0,52134.0,32725.0,13459.0,0.0,207494.0
91,2025-09-19,2025,sep,9,19,90452.0,33229.0,18902.0,2618.0,0.0,145201.0
90,2025-09-19,2025,sep,9,19,NaN,NaN,NaN,NaN,NaN,NaN
115,2025-09-23,2025,sep,9,23,NaN,NaN,NaN,NaN,NaN,NaN


In [108]:
cols_num = ["lima", "sierra", "costa_norte", "selva", "costa_sur", "total"]

# Convertir columnas numéricas por seguridad
for c in cols_num:
    df_total_tabla2[c] = pd.to_numeric(df_total_tabla2[c], errors="coerce")

# 1. Eliminar filas totalmente vacías en valores numéricos
df_total_tabla2 = df_total_tabla2[
    df_total_tabla2[cols_num].notna().any(axis=1)
].copy()

# 2. Si hay duplicados por fecha, quedarse con la fila más completa
df_total_tabla2["n_no_nulos"] = df_total_tabla2[cols_num].notna().sum(axis=1)

df_total_tabla2 = (
    df_total_tabla2
    .sort_values(["fecha", "n_no_nulos"], ascending=[True, False])
    .drop_duplicates(subset=["fecha"], keep="first")
    .drop(columns="n_no_nulos")
    .sort_values("fecha")
    .reset_index(drop=True)
)

# 3. Guardar histórico limpio
df_total_tabla2.to_csv("tabla_2.csv", index=False)

df_total_tabla2

,fecha,anio,mes_txt,mes_num,dia,lima,sierra,costa_norte,selva,costa_sur,total
0,2025-09-03,2025,sep,9,3,286212.0,115589.0,0.0,10368.0,6149.0,418318.0
1,2025-09-04,2025,sep,9,4,156664.0,53435.0,0.0,3040.0,15899.0,229038.0
2,2025-09-05,2025,sep,9,5,103191.0,62557.0,0.0,9598.0,9944.0,185290.0
3,2025-09-06,2025,sep,9,6,97955.0,72622.0,0.0,6363.0,4969.0,181909.0
4,2025-09-07,2025,sep,9,7,24295.0,0.0,0.0,0.0,0.0,24295.0
...,...,...,...,...,...,...,...,...,...,...,...
205,2026-04-18,2026,abr,4,18,100437.0,65127.0,20421.0,6263.0,0.0,192249.0
206,2026-04-19,2026,abr,4,19,28155.0,0.0,0.0,0.0,0.0,28155.0
207,2026-04-20,2026,abr,4,20,149288.0,127808.0,40762.0,6150.0,0.0,324008.0
208,2026-04-21,2026,abr,4,21,136093.0,121187.0,6215.0,11891.0,0.0,275386.0


In [109]:
# ver si hay fechas duplicadas
df_total_tabla2[df_total_tabla2.duplicated(subset=["fecha"], keep=False)].sort_values("fecha")

,fecha,anio,mes_txt,mes_num,dia,lima,sierra,costa_norte,selva,costa_sur,total


In [110]:
# ahora si la descargarmos como tabla_2.csv
df_total_tabla2.to_csv("tabla_2.csv", index=False)